In [78]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters.character import CharacterTextSplitter
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_chroma import Chroma
import re
import copy

In [95]:
# 👉 1. ADD MULTIPLE PDF FILES HERE
pdf_files = [
    "SERVICE _ WARRANTY POLICY.pdf",
    "TERMS and conditions.pdf",
    "RETURNS _ REFUNDS POLICY.pdf",
    "Bulk qutation policy.pdf",
    "PRIVACY POLICY.pdf"
]

all_pages = []


In [96]:
# 👉 2. LOAD ALL PDFs
for file in pdf_files:
    loader = PyPDFLoader(file)
    pages = loader.load()
    
    # Add source info (VERY IMPORTANT for multi-doc)
    for p in pages:
        p.metadata["source_file"] = file
    
    all_pages.extend(pages)

In [97]:
all_pages

[Document(metadata={'producer': 'Microsoft® Word LTSC', 'creator': 'Microsoft® Word LTSC', 'creationdate': '2025-01-09T01:54:55+05:00', 'author': 'Ok', 'moddate': '2025-01-09T01:54:55+05:00', 'source': 'SERVICE _ WARRANTY POLICY.pdf', 'total_pages': 14, 'page': 0, 'page_label': '1', 'source_file': 'SERVICE _ WARRANTY POLICY.pdf'}, page_content='1 \n \nSERVICE & WARRANTY POLICY \nEffective Date: ___________________________. \nLast Updated: ___________________________. \nINTRODUCTION \nAt Star Gallery Mart , we are committed to providing high -quality products backed by robust \nwarranty support. We understand the importance of ensuring our customers feel confident in their \npurchases, which is why we offer comprehensive warranty coverage on branded items to safeguard \nagainst manufacturing defects. This policy outlines the warranty terms, exclusions, claim \nprocedures, and important details on repairs or replacements. \nOur goal is to provide efficient and customer-friendly solutions

In [99]:
# 👉 3. CLEAN TEXT 
pages_pdf_clean = copy.deepcopy(all_pages)

for doc in pages_pdf_clean:
    text = doc.page_content
    text = " ".join(text.split())
    doc.page_content = text

In [100]:
pages_pdf_clean

[Document(metadata={'producer': 'Microsoft® Word LTSC', 'creator': 'Microsoft® Word LTSC', 'creationdate': '2025-01-09T01:54:55+05:00', 'author': 'Ok', 'moddate': '2025-01-09T01:54:55+05:00', 'source': 'SERVICE _ WARRANTY POLICY.pdf', 'total_pages': 14, 'page': 0, 'page_label': '1', 'source_file': 'SERVICE _ WARRANTY POLICY.pdf'}, page_content='1 SERVICE & WARRANTY POLICY Effective Date: ___________________________. Last Updated: ___________________________. INTRODUCTION At Star Gallery Mart , we are committed to providing high -quality products backed by robust warranty support. We understand the importance of ensuring our customers feel confident in their purchases, which is why we offer comprehensive warranty coverage on branded items to safeguard against manufacturing defects. This policy outlines the warranty terms, exclusions, claim procedures, and important details on repairs or replacements. Our goal is to provide efficient and customer-friendly solutions for products that may 

In [101]:
# 👉 4. SPLIT
char_splitter = CharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

pages_char_split = char_splitter.split_documents(pages_pdf_clean)


In [102]:
# 👉 5. EMBEDDINGS
embeddings_model = OpenAIEmbeddings(model="text-embedding-3-small")

In [104]:
# 👉 6. STORE IN CHROMA (same directory — supports multiple PDFs)
vectorstore = Chroma.from_documents(
    documents=pages_char_split,
    embedding=embeddings_model,
    persist_directory="./chroma_db"
)

In [105]:
print(f"Stored {len(pages_char_split)} chunks from {len(pdf_files)} PDFs!")


Stored 63 chunks from 5 PDFs!


In [ ]:
# 👉 7. QUERY
vectorstore = Chroma(
    embedding_function=embeddings_model,
    persist_directory="./chroma_db"
)

results = vectorstore.similarity_search(
    "return duration",
    k=3
)

for r in results:
    print("SOURCE:", r.metadata.get("source"))
    print(r.page_content)
    print("------")